# Notebook Title: Exploratory Data Analysis (EDA) - Student & Career Profiles

## Purpose
"Exploratory Data Analysis of raw and refined student profiles" 
To validate data integrity, identify underlying distributions across the 11-point vector space, and confirm the statistical readiness of the datasets for Machine Learning model training.

## Context
- **Project Phase:** Exploration / Research
- **Related Module(s):** `src/data_processing`, `src/research`

## Data Sources
- **External**
- **Raw Kaggle Dataset:** Student personality survey responses (TIPI and RIASEC).
- **Refined Vector Library:** Normalized 11-point vector profiles (6 RIASEC + 5 Big Five) derived from the raw Kaggle data and O*NET Career Work Styles.

## Assumptions & Constraints
- **Assumptions:** Survey responses are considered accurate reflections of user traits; O*NET mappings to Big Five are psychologically sound.
- **Constraints:** Analysis is limited to the 11 dimensions defined in the unified vector space; original Kaggle data may contain noise or missing values that require filtering.

## Reproducibility
- **Environment:** Local (VS Code / Jupyter / Anaconda)

## Expected Outputs
- **Distribution Plots:** Histograms showing the spread of RIASEC and Big Five scores.
- **Correlation Heatmap:** Visualizing the relationship between interests and personality traits.
- **PCA Visualization:** 2D mapping of the 11D vector space to identify clusters and data coverage.
- **Data Health Metrics:** Missing value counts and normalization range checks (0.0 to 1.0).

## Notes
- This notebook acts as a "Data Health Check" to ensure that the normalization and feature alignment between the Kaggle (Student) and O*NET (Career) datasets are mathematically consistent before training the Random Forest model.

In [7]:
import pandas as pd

# Update this path to where your Kaggle Raw file is saved
kaggle_raw_path = 'docs\data\Kaggle_Raw_Majors.csv'

try:
    kaggle_raw = pd.read_csv(kaggle_raw_path)
except FileNotFoundError:
    kaggle_raw = pd.read_csv(f"../{kaggle_raw_path}")

print("Kaggle Columns:")
print(kaggle_raw.columns.tolist())

# Look at the first 5 rows to see how they store the RIASEC data
display(kaggle_raw.head())

<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
C:\Users\grosh\AppData\Local\Temp\ipykernel_70168\1105363794.py:4: SyntaxWarning: invalid escape sequence '\d'
  kaggle_raw_path = 'docs\data\Kaggle_Raw_Majors.csv'


Kaggle Columns:
['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8', 'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'introelapse', 'testelapse', 'surveyelapse', 'TIPI1', 'TIPI2', 'TIPI3', 'TIPI4', 'TIPI5', 'TIPI6', 'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10', 'VCL1', 'VCL2', 'VCL3', 'VCL4', 'VCL5', 'VCL6', 'VCL7', 'VCL8', 'VCL9', 'VCL10', 'VCL11', 'VCL12', 'VCL13', 'VCL14', 'VCL15', 'VCL16', 'education', 'urban', 'gender', 'engnat', 'age', 'hand', 'religion', 'orientation', 'race', 'voted', 'married', 'familysize', 'uniqueNetworkLocation', 'country', 'source', 'major']


,R1,R2,R3,R4,R5,R6,R7,R8,I1,I2,...,religion,orientation,race,voted,married,familysize,uniqueNetworkLocation,country,source,major
0,1,1,2,4,1,2,2,1,5,5,...,7,3,4,1,2,3,1,US,1,Nursing
1,4,1,1,2,1,1,1,2,5,5,...,4,3,1,2,1,4,1,PH,0,education
2,3,5,1,3,1,5,3,4,4,5,...,2,1,5,1,1,2,1,IN,2,Literature
3,1,4,1,4,1,4,1,2,4,4,...,2,1,1,2,1,3,2,US,0,Math
4,5,1,2,2,2,1,2,1,4,4,...,7,3,1,2,1,0,1,PH,0,mathematics and science


In [8]:
kaggle_raw.describe(include='all')

,R1,R2,R3,R4,R5,R6,R7,R8,I1,I2,...,religion,orientation,race,voted,married,familysize,uniqueNetworkLocation,country,source,major
count,92954.000000,92954.000000,92954.000000,92954.000000,92954.000000,92954.000000,92954.000000,92954.000000,92954.000000,92954.000000,...,92954.000000,92954.000000,92954.000000,92954.000000,92954.000000,9.295400e+04,92954.000000,92948,92954.000000,92954
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,182,NaN,15953
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,US,NaN,psychology
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50449,NaN,6861
mean,2.599286,2.159240,1.767197,2.275276,1.747596,2.271984,2.015847,1.997547,3.495256,3.373432,...,5.551563,1.348613,3.249457,1.524496,1.351615,1.955833e+05,1.246735,NaN,0.416529,NaN
std,1.331454,1.238005,1.129459,1.322731,1.063415,1.280006,1.173389,1.181730,1.307994,1.349221,...,3.409146,0.987252,1.400614,0.522129,0.604330,2.018972e+07,0.431113,NaN,0.646567,NaN
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,1.000000,NaN,0.000000,NaN
25%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3.000000,2.000000,...,2.000000,1.000000,2.000000,1.000000,1.000000,2.000000e+00,1.000000,NaN,0.000000,NaN
50%,3.000000,2.000000,1.000000,2.000000,1.000000,2.000000,2.000000,2.000000,4.000000,4.000000,...,6.000000,1.000000,4.000000,2.000000,1.000000,3.000000e+00,1.000000,NaN,0.000000,NaN
75%,4.000000,3.000000,2.000000,3.000000,2.000000,3.000000,3.000000,3.000000,5.000000,4.000000,...,7.000000,1.000000,4.000000,2.000000,2.000000,3.000000e+00,1.000000,NaN,1.000000,NaN


In [9]:
kaggle_raw.isnull().sum()

R1                       0
R2                       0
R3                       0
R4                       0
R5                       0
                        ..
familysize               0
uniqueNetworkLocation    0
country                  6
source                   0
major                    0
Length: 93, dtype: int64

In [4]:
kaggle_raw.columns

Index(['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8', 'I1', 'I2', 'I3', 'I4',
       'I5', 'I6', 'I7', 'I8', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8',
       'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'E1', 'E2', 'E3', 'E4',
       'E5', 'E6', 'E7', 'E8', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8',
       'introelapse', 'testelapse', 'surveyelapse', 'TIPI1', 'TIPI2', 'TIPI3',
       'TIPI4', 'TIPI5', 'TIPI6', 'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10', 'VCL1',
       'VCL2', 'VCL3', 'VCL4', 'VCL5', 'VCL6', 'VCL7', 'VCL8', 'VCL9', 'VCL10',
       'VCL11', 'VCL12', 'VCL13', 'VCL14', 'VCL15', 'VCL16', 'education',
       'urban', 'gender', 'engnat', 'age', 'hand', 'religion', 'orientation',
       'race', 'voted', 'married', 'familysize', 'uniqueNetworkLocation',
       'country', 'source', 'major'],
      dtype='object')